# ImageNet-C: BMIA vs Baselines — AAAI 2027
**Methods**: Source, TENT, EATA, BMIA-Fixed, BMIA-Adaptive  
**Dataset**: ImageNet validation + imagecorruptions (generated on the fly)  
**Model**: ResNet-50 pretrained on ImageNet  
**Corruptions**: all 15, severity 5  
**LRs**: 0.005, 0.05

## Cell 1 — Install dependencies

In [ ]:
!pip install imagecorruptions scipy -q

## Cell 2 — Imports

In [ ]:
import os, csv, time, warnings
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from imagecorruptions import corrupt

warnings.filterwarnings('ignore')
print('Torch  :', torch.__version__)
print('CUDA   :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU    :', torch.cuda.get_device_name(0))

## Cell 3 — Configuration

In [ ]:
# ImageNet validation path on Kaggle
IMAGENET_VAL = '/kaggle/input/imagenet-object-localization-challenge/ILSVRC/Data/CLS-LOC/val'

RESULTS_DIR  = '/kaggle/working/results_imagenet_c'
SEVERITY     = 5
BATCH_SIZE   = 64
SEED         = 42
NUM_WORKERS  = 2
LEARNING_RATES = [0.005, 0.05]
K            = 1000

# 5000 images = 10% of ImageNet val, statistically meaningful (~0.6% std error)
IMAGES_PER_CORRUPTION = 5000

CORRUPTIONS = [
    'gaussian_noise', 'shot_noise',    'impulse_noise',
    'defocus_blur',   'glass_blur',    'motion_blur',   'zoom_blur',
    'snow',           'frost',         'fog',
    'brightness',     'contrast',      'elastic_transform',
    'pixelate',       'jpeg_compression',
]

# EATA hyperparams (from paper: E0=0.4*log(K), weight=2000)
E0            = 0.4 * np.log(K)   # 2.763 nats
FISHER_WEIGHT = 2000.0
FISHER_N      = 500               # samples for Fisher estimation

# BMIA hyperparams (same as paper)
LAMBDA_FIXED = 1.0
LAMBDA_INIT  = 0.5
LAMBDA_MIN   = 0.01
LAMBDA_MAX   = 10.0
# TAU = 0.10 (same as paper's validated value)
# NOTE: with batch=64 and K=1000, random dom% ≈ 1.5%, so TAU=0.10 only
# triggers on real concentration — correct behavior.
# (TAU=10/K=0.01 would fire every batch — wrong)
TAU = 0.10

# Collapse criterion for K=1000 (paper formula: 2/K AND active < K/2)
COLLAPSE_DOM_THRESH    = 2.0 / K   # 0.002 (0.2%)
COLLAPSE_ACTIVE_THRESH = K // 2    # 500 classes
COLLAPSE_HARD_THRESH   = K // 5    # 200 classes (hard collapse)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(SEED)
np.random.seed(SEED)
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f'Device : {device}')
print(f'Val dir: {IMAGENET_VAL}')
print(f'Exists : {os.path.isdir(IMAGENET_VAL)}')

## Cell 4 — Build val label map
ImageNet val images are flat (no subfolders). We need a label file.

In [ ]:
# Kaggle ImageNet val labels
LABEL_FILE = '/kaggle/input/imagenet-object-localization-challenge/LOC_val_solution.csv'

# ── Auto-discovery: find the actual mounted path if default doesn't exist ──
import os as _os

def _find_file(filename, search_roots):
    for root in search_roots:
        if not _os.path.isdir(root):
            continue
        for dirpath, _, files in _os.walk(root):
            if filename in files:
                return _os.path.join(dirpath, filename)
    return None

if not _os.path.exists(LABEL_FILE):
    print(f'[WARN] Default label path not found. Searching /kaggle/input/ ...')
    found = _find_file('LOC_val_solution.csv', ['/kaggle/input'])
    if found:
        LABEL_FILE = found
        print(f'[OK]   Found label file at: {LABEL_FILE}')
    else:
        print('[ERROR] LOC_val_solution.csv not found anywhere in /kaggle/input/')
        print('        Please add the "ImageNet Object Localization Challenge" dataset')
        print('        via the Data panel → + Add Data in your Kaggle notebook.')
        raise FileNotFoundError('LOC_val_solution.csv not found — dataset not added')
else:
    print(f'[OK]   Label file: {LABEL_FILE}')

# Also auto-discover val image directory
IMAGENET_VAL = '/kaggle/input/imagenet-object-localization-challenge/ILSVRC/Data/CLS-LOC/val'
if not _os.path.isdir(IMAGENET_VAL):
    print(f'[WARN] Default val path not found. Searching for val directory ...')
    # Try to find any directory named 'val' under imagenet input
    for root in ['/kaggle/input']:
        for dirpath, dirs, _ in _os.walk(root):
            if 'val' in dirs:
                candidate = _os.path.join(dirpath, 'val')
                # Must contain .JPEG files
                sample = next((_os.path.join(candidate, f)
                               for f in _os.listdir(candidate)
                               if f.endswith('.JPEG')), None)
                if sample:
                    IMAGENET_VAL = candidate
                    print(f'[OK]   Found val dir at: {IMAGENET_VAL}')
                    break
            if _os.path.isdir(IMAGENET_VAL) and IMAGENET_VAL != '/kaggle/input/imagenet-object-localization-challenge/ILSVRC/Data/CLS-LOC/val':
                break
    if not _os.path.isdir(IMAGENET_VAL):
        raise FileNotFoundError(f'Val image directory not found: {IMAGENET_VAL}')
else:
    print(f'[OK]   Val images: {IMAGENET_VAL}')


def build_label_map():
    """
    Returns list of (image_path, class_idx) tuples.
    synset_to_idx is alphabetically sorted — matches torchvision pretrained ordering.
    """
    import csv as _csv

    synsets = set()
    rows    = {}
    with open(LABEL_FILE, 'r') as f:
        reader = _csv.DictReader(f)
        for row in reader:
            img_id = row['ImageId']                       # ILSVRC2012_val_00000001
            synset = row['PredictionString'].split()[0]   # first token = ground-truth synset
            synsets.add(synset)
            rows[img_id] = synset

    # Alphabetical sort matches torchvision ResNet-50 class ordering
    synset_to_idx = {s: i for i, s in enumerate(sorted(synsets))}

    img_list = []
    for img_id, synset in sorted(rows.items()):
        path = os.path.join(IMAGENET_VAL, img_id + '.JPEG')
        if os.path.exists(path):
            img_list.append((path, synset_to_idx[synset]))

    print(f'Total val images found : {len(img_list)}')
    print(f'Total classes          : {len(synset_to_idx)}')
    return img_list, synset_to_idx

img_list, synset_to_idx = build_label_map()

# Fixed random subset — same images used across all corruptions and methods
np.random.seed(SEED)
indices    = np.random.choice(len(img_list), IMAGES_PER_CORRUPTION, replace=False)
img_subset = [img_list[i] for i in sorted(indices)]
print(f'Using subset           : {len(img_subset)} images')

## Cell 5 — Corrupted Dataset

In [ ]:
NORMALIZE = transforms.Normalize(
    mean=[0.485, 0.456, 0.406],
    std =[0.229, 0.224, 0.225]
)
TO_TENSOR = transforms.Compose([
    transforms.ToTensor(),
    NORMALIZE,
])
RESIZE_CROP = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
])

class CorruptedImageNet(Dataset):
    """
    Applies imagecorruptions on-the-fly.
    Reads PIL image -> resize/crop -> numpy -> corrupt -> tensor.
    """
    def __init__(self, img_list, corruption_name, severity):
        self.img_list        = img_list
        self.corruption_name = corruption_name
        self.severity        = severity

    def __len__(self):
        return len(self.img_list)

    def __getitem__(self, idx):
        path, label = self.img_list[idx]
        img = Image.open(path).convert('RGB')
        img = RESIZE_CROP(img)                      # PIL 224x224
        img_np = np.array(img, dtype=np.uint8)      # HxWx3 uint8
        try:
            img_np = corrupt(img_np,
                             corruption_name=self.corruption_name,
                             severity=self.severity)
        except Exception:
            pass  # if corruption fails, use clean image
        img_pil = Image.fromarray(img_np.astype(np.uint8))
        tensor  = TO_TENSOR(img_pil)
        return tensor, label


def get_loader(corruption):
    ds = CorruptedImageNet(img_subset, corruption, SEVERITY)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False,
                      num_workers=NUM_WORKERS, pin_memory=True)

# Quick sanity check
print('Testing dataset...')
test_ds = CorruptedImageNet(img_subset[:4], 'gaussian_noise', 5)
img, lbl = test_ds[0]
print(f'Image shape: {img.shape}  Label: {lbl}  dtype: {img.dtype}')
print('Dataset OK')

## Cell 6 — Model Utilities

In [ ]:
def load_resnet50():
    try:
        m = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
    except AttributeError:
        m = models.resnet50(pretrained=True)
    return m.to(device)


def configure_for_tta(model):
    """BN layers: train mode (batch stats), only affine params trainable."""
    model.eval()
    for p in model.parameters():
        p.requires_grad_(False)
    for m in model.modules():
        if isinstance(m, nn.BatchNorm2d):
            m.train()
            m.requires_grad_(True)
            m.track_running_stats = False
            m.running_mean = None
            m.running_var  = None
    return model


def get_bn_params(model):
    return [p for m in model.modules()
            if isinstance(m, nn.BatchNorm2d)
            for p in m.parameters() if p.requires_grad]


def save_theta0(model):
    t0 = {}
    for nm, m in model.named_modules():
        if isinstance(m, nn.BatchNorm2d):
            t0[nm + '.w'] = m.weight.data.clone()
            t0[nm + '.b'] = m.bias.data.clone()
    return t0


print('Model utilities ready.')

## Cell 7 — Loss Functions

In [ ]:
def entropy_per_sample(outputs):
    """H(Y|x) per sample. Shape: (B,)"""
    p = torch.softmax(outputs, dim=1).clamp(min=1e-8)
    return -(p * p.log()).sum(dim=1)


def mi_loss(outputs):
    """-I(X;Y) = H(Y|X) - H(Y).  Minimize = maximize MI."""
    p   = torch.softmax(outputs, dim=1).clamp(min=1e-8)
    hyx = -(p * p.log()).sum(dim=1).mean()
    q   = p.mean(dim=0).clamp(min=1e-8)
    hy  = -(q * q.log()).sum()
    return hyx - hy


def anchor_loss(model, theta0):
    """||theta - theta0||^2 over BN affine params."""
    loss = torch.tensor(0.0, device=device)
    for nm, m in model.named_modules():
        if isinstance(m, nn.BatchNorm2d):
            loss = loss + (m.weight - theta0[nm + '.w']).pow(2).sum()
            loss = loss + (m.bias   - theta0[nm + '.b']).pow(2).sum()
    return loss


print('Loss functions ready.')

## Cell 8 — Metrics

In [ ]:
def compute_metrics(all_preds, all_labels, hyx_sum, p_sum, n_total):
    preds  = torch.cat(all_preds)
    labels = torch.cat(all_labels)
    acc    = preds.eq(labels).float().mean().item() * 100.0

    counts  = torch.bincount(preds, minlength=K)
    dom_pct = counts.max().item() / preds.shape[0]
    active  = (counts > 0).sum().item()
    collapsed = int(
        (dom_pct > COLLAPSE_DOM_THRESH and active < COLLAPSE_ACTIVE_THRESH)
        or active < COLLAPSE_HARD_THRESH
    )

    hyx = hyx_sum / n_total
    q   = (p_sum / n_total).clamp(min=1e-8)
    hy  = -(q * q.log()).sum().item()
    mi  = max(hy - hyx, 0.0)

    return dict(
        acc=round(acc, 2),
        collapsed=collapsed,
        dom_pct=round(dom_pct * 100, 3),
        hy=round(hy, 4),
        hyx=round(hyx, 4),
        mi=round(mi, 4),
        active_classes=active,
    )


print('Metrics ready.')

## Cell 9 — Fisher Estimation (EATA)

In [ ]:
def estimate_fisher(model, loader):
    fisher = {}
    for nm, m in model.named_modules():
        if isinstance(m, nn.BatchNorm2d):
            fisher[nm + '.w'] = torch.zeros_like(m.weight)
            fisher[nm + '.b'] = torch.zeros_like(m.bias)

    model.zero_grad()
    n_seen = 0
    for images, _ in loader:
        if n_seen >= FISHER_N:
            break
        images = images.to(device)
        bsz    = images.shape[0]
        loss   = entropy_per_sample(model(images)).mean()
        loss.backward()
        for nm, m in model.named_modules():
            if isinstance(m, nn.BatchNorm2d):
                if m.weight.grad is not None:
                    fisher[nm + '.w'] += m.weight.grad.data.pow(2) * bsz
                if m.bias.grad is not None:
                    fisher[nm + '.b'] += m.bias.grad.data.pow(2) * bsz
        model.zero_grad()
        n_seen += bsz

    n_seen = max(n_seen, 1)
    for k in fisher:
        fisher[k] /= n_seen
    return fisher


print('Fisher estimation ready.')

## Cell 10 — TTA Methods

In [ ]:
def run_source(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    hyx_sum, p_sum, n_total = 0.0, torch.zeros(K), 0
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outs   = model(images)
            p = torch.softmax(outs, dim=1).clamp(min=1e-8).cpu()
            hyx_sum += -(p * p.log()).sum(dim=1).sum().item()
            p_sum   += p.sum(dim=0)
            n_total += len(labels)
            all_preds.append(outs.argmax(1).cpu())
            all_labels.append(labels)
    return compute_metrics(all_preds, all_labels, hyx_sum, p_sum, n_total)


def run_tent(model, loader, lr):
    configure_for_tta(model)
    opt = torch.optim.SGD(get_bn_params(model), lr=lr, momentum=0.9)
    all_preds, all_labels = [], []
    hyx_sum, p_sum, n_total = 0.0, torch.zeros(K), 0
    for images, labels in loader:
        images = images.to(device)
        opt.zero_grad()
        outs = model(images)
        entropy_per_sample(outs).mean().backward()
        opt.step()
        with torch.no_grad():
            p = torch.softmax(outs, dim=1).clamp(min=1e-8).cpu().detach()
        hyx_sum += -(p * p.log()).sum(dim=1).sum().item()
        p_sum   += p.sum(dim=0)
        n_total += len(labels)
        all_preds.append(outs.argmax(1).cpu().detach())
        all_labels.append(labels)
    return compute_metrics(all_preds, all_labels, hyx_sum, p_sum, n_total)


def run_eata(model, loader, lr):
    configure_for_tta(model)
    theta0 = save_theta0(model)
    fisher = estimate_fisher(model, loader)
    opt    = torch.optim.SGD(get_bn_params(model), lr=lr, momentum=0.9)
    all_preds, all_labels = [], []
    hyx_sum, p_sum, n_total = 0.0, torch.zeros(K), 0
    for images, labels in loader:
        images = images.to(device)
        opt.zero_grad()
        outs = model(images)
        ent  = entropy_per_sample(outs)
        mask = ent < E0
        if mask.sum() > 0:
            reg = torch.tensor(0.0, device=device)
            for nm, m in model.named_modules():
                if isinstance(m, nn.BatchNorm2d):
                    reg = reg + (fisher[nm+'.w'] * (m.weight - theta0[nm+'.w']).pow(2)).sum()
                    reg = reg + (fisher[nm+'.b'] * (m.bias   - theta0[nm+'.b']).pow(2)).sum()
            loss = ent[mask].mean() + (FISHER_WEIGHT / 2.0) * reg
            loss.backward()
            opt.step()
        with torch.no_grad():
            p = torch.softmax(outs, dim=1).clamp(min=1e-8).cpu().detach()
        hyx_sum += -(p * p.log()).sum(dim=1).sum().item()
        p_sum   += p.sum(dim=0)
        n_total += len(labels)
        all_preds.append(outs.argmax(1).cpu().detach())
        all_labels.append(labels)
    return compute_metrics(all_preds, all_labels, hyx_sum, p_sum, n_total)


def run_bmia_fixed(model, loader, lr):
    configure_for_tta(model)
    theta0 = save_theta0(model)
    opt    = torch.optim.SGD(get_bn_params(model), lr=lr, momentum=0.9)
    all_preds, all_labels = [], []
    hyx_sum, p_sum, n_total = 0.0, torch.zeros(K), 0
    for images, labels in loader:
        images = images.to(device)
        opt.zero_grad()
        outs = model(images)
        loss = mi_loss(outs) + LAMBDA_FIXED * anchor_loss(model, theta0)
        loss.backward()
        opt.step()
        with torch.no_grad():
            p = torch.softmax(outs, dim=1).clamp(min=1e-8).cpu().detach()
        hyx_sum += -(p * p.log()).sum(dim=1).sum().item()
        p_sum   += p.sum(dim=0)
        n_total += len(labels)
        all_preds.append(outs.argmax(1).cpu().detach())
        all_labels.append(labels)
    return compute_metrics(all_preds, all_labels, hyx_sum, p_sum, n_total)


def run_bmia_adaptive(model, loader, lr):
    configure_for_tta(model)
    theta0 = save_theta0(model)
    opt    = torch.optim.SGD(get_bn_params(model), lr=lr, momentum=0.9)
    lam    = LAMBDA_INIT
    all_preds, all_labels = [], []
    hyx_sum, p_sum, n_total = 0.0, torch.zeros(K), 0
    for images, labels in loader:
        images = images.to(device)
        opt.zero_grad()
        outs = model(images)
        loss = mi_loss(outs) + lam * anchor_loss(model, theta0)
        loss.backward()
        opt.step()
        with torch.no_grad():
            preds  = outs.argmax(1)
            counts = torch.bincount(preds, minlength=K)
            dom    = counts.max().item() / preds.shape[0]
        lam = min(LAMBDA_MAX, lam * 1.1) if dom > TAU else max(LAMBDA_MIN, lam * 0.95)
        with torch.no_grad():
            p = torch.softmax(outs, dim=1).clamp(min=1e-8).cpu().detach()
        hyx_sum += -(p * p.log()).sum(dim=1).sum().item()
        p_sum   += p.sum(dim=0)
        n_total += len(labels)
        all_preds.append(preds.cpu())
        all_labels.append(labels)
    return compute_metrics(all_preds, all_labels, hyx_sum, p_sum, n_total)


print('All TTA methods ready.')

## Cell 11 — Main Experiment Loop
15 corruptions × 2 LRs × 4 methods + Source = 125 runs  
Results saved to CSV after every single run (safe against interruption).

In [ ]:
FIELDS = ['corruption','lr','method','acc','collapsed',
          'dom_pct','hy','hyx','mi','active_classes','time_s']

results_file = os.path.join(RESULTS_DIR, 'imagenet_c_results.csv')
with open(results_file, 'w', newline='') as f:
    csv.DictWriter(f, fieldnames=FIELDS).writeheader()

TTA_METHODS  = ['TENT', 'EATA', 'BMIA-Fixed', 'BMIA-Adaptive']
total_start  = time.time()

for corruption in CORRUPTIONS:
    print(f'\n{"="*65}')
    print(f'  Corruption: {corruption}')
    print(f'{"="*65}')
    loader = get_loader(corruption)

    # Source — no lr dependency, run once
    t0    = time.time()
    model = load_resnet50()
    src   = run_source(model, loader)
    del model
    elapsed = round(time.time() - t0, 1)
    print(f"  {'Source':20s} lr=N/A  acc={src['acc']:.1f}%  "
          f"col={src['collapsed']}  MI={src['mi']:.3f}  [{elapsed}s]")
    with open(results_file, 'a', newline='') as f:
        csv.DictWriter(f, fieldnames=FIELDS).writerow(
            dict(corruption=corruption, lr='N/A', method='Source', **src, time_s=elapsed))

    # TTA methods
    for lr in LEARNING_RATES:
        for method in TTA_METHODS:
            t0    = time.time()
            model = load_resnet50()
            if   method == 'TENT':           metrics = run_tent(model, loader, lr)
            elif method == 'EATA':           metrics = run_eata(model, loader, lr)
            elif method == 'BMIA-Fixed':     metrics = run_bmia_fixed(model, loader, lr)
            elif method == 'BMIA-Adaptive':  metrics = run_bmia_adaptive(model, loader, lr)
            del model
            elapsed = round(time.time() - t0, 1)
            print(f"  {method:20s} lr={lr}  acc={metrics['acc']:.1f}%  "
                  f"col={metrics['collapsed']}  MI={metrics['mi']:.3f}  [{elapsed}s]")
            with open(results_file, 'a', newline='') as f:
                csv.DictWriter(f, fieldnames=FIELDS).writerow(
                    dict(corruption=corruption, lr=lr, method=method,
                         **metrics, time_s=elapsed))

total_time = round(time.time() - total_start)
print(f'\nDONE. Total time: {total_time}s  |  Results: {results_file}')

## Cell 12 — Results Summary

In [ ]:
import pandas as pd

df = pd.read_csv(results_file)

LOG_K = float(np.log(K))   # CSD Theorem: MI upper bound = log K nats

print('='*70)
print('IMAGENET-C RESULTS — Avg over 15 Corruptions, Severity 5')
print(f'Subset: {IMAGES_PER_CORRUPTION} images, ResNet-50')
print(f'CSD Theorem  — MI upper bound log(K={K}): {LOG_K:.3f} nats  |  Collapse MI: 0.000')
print('='*70)

for lr in LEARNING_RATES:
    sub = df[df['lr'].astype(str) == str(lr)]
    if sub.empty: continue
    print(f'\n── lr = {lr} ──────────────────────────────────')
    g = sub.groupby('method').agg(
        Acc=('acc','mean'),
        Collapses=('collapsed','sum'),
        MI=('mi','mean')
    ).round(2)
    print(g.to_string())

src_df = df[df['method'] == 'Source']
print(f'\n── Source ──────────────────────────────────────')
print(f"  Avg acc: {src_df['acc'].mean():.2f}%  |  MI: {src_df['mi'].mean():.3f}")

print('\n── Collapse counts (out of 15 runs per lr) ─────')
for lr in LEARNING_RATES:
    sub = df[(df['lr'].astype(str)==str(lr)) & (df['method']!='Source')]
    if sub.empty: continue
    print(f'\n  lr={lr}:')
    for m, c in sub.groupby('method')['collapsed'].sum().items():
        print(f'    {m:22s}: {int(c):2d}/15')

## Cell 13 — Per-Corruption Table (paper-ready)

In [ ]:
import pandas as pd

df = pd.read_csv(results_file)

for lr in LEARNING_RATES:
    sub = df[df['lr'].astype(str) == str(lr)]
    if sub.empty: continue
    print(f'\n── Per-Corruption Accuracy (lr={lr}) ───────────')
    pivot = sub.pivot_table(index='corruption', columns='method', values='acc')
    src   = df[df['method']=='Source'].set_index('corruption')['acc']
    pivot.insert(0, 'Source', src)
    cols  = ['Source','TENT','EATA','BMIA-Fixed','BMIA-Adaptive']
    pivot = pivot[[c for c in cols if c in pivot.columns]]
    print(pivot.round(1).to_string())
    avgs  = pivot.mean().round(1)
    print(f'\n  AVG : {dict(avgs)}')